In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

# Sửa đường dẫn này nếu bạn để file ở chỗ khác
MESSAGES_JSONL_PATH = Path(r"C:\Users\Admin\Chatbot_answering_vietnamese_history\Dataset\Pack1\bach_dang_ngo_quyen_rag_sft_messages_20.jsonl")

print("File path:", MESSAGES_JSONL_PATH)
print("Exists:", MESSAGES_JSONL_PATH.exists())

if MESSAGES_JSONL_PATH.exists():
    print("Size:", f"{MESSAGES_JSONL_PATH.stat().st_size / 1024:.2f} KB")
else:
    print("Không tìm thấy file. Hãy kiểm tra lại đường dẫn.")

File path: C:\Users\Admin\Chatbot_answering_vietnamese_history\Dataset\Pack1\bach_dang_ngo_quyen_rag_sft_messages_20.jsonl
Exists: True
Size: 156.84 KB


In [2]:
records = []
bad_lines = []

with open(MESSAGES_JSONL_PATH, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        line = line.strip()

        if not line:
            continue

        try:
            obj = json.loads(line)
            records.append(obj)
        except Exception as e:
            bad_lines.append({
                "line_no": line_no,
                "error": str(e),
                "preview": line[:300],
            })

print("Đọc được records:", len(records))
print("Bad lines:", len(bad_lines))

if bad_lines:
    display(pd.DataFrame(bad_lines).head(10))

Đọc được records: 20
Bad lines: 0


In [3]:
schema_errors = []

for i, r in enumerate(records):
    sample_id = r.get("id", f"row_{i}")

    if "messages" not in r:
        schema_errors.append({
            "row": i,
            "id": sample_id,
            "error": "Thiếu field messages",
        })
        continue

    messages = r["messages"]

    if not isinstance(messages, list):
        schema_errors.append({
            "row": i,
            "id": sample_id,
            "error": "messages không phải list",
        })
        continue

    if len(messages) < 2:
        schema_errors.append({
            "row": i,
            "id": sample_id,
            "error": "messages có ít hơn 2 message",
        })
        continue

    for j, msg in enumerate(messages):
        if not isinstance(msg, dict):
            schema_errors.append({
                "row": i,
                "id": sample_id,
                "error": f"message {j} không phải dict",
            })
            continue

        if "role" not in msg:
            schema_errors.append({
                "row": i,
                "id": sample_id,
                "error": f"message {j} thiếu role",
            })

        if "content" not in msg:
            schema_errors.append({
                "row": i,
                "id": sample_id,
                "error": f"message {j} thiếu content",
            })

        if msg.get("role") not in ["system", "user", "assistant"]:
            schema_errors.append({
                "row": i,
                "id": sample_id,
                "error": f"message {j} có role lạ: {msg.get('role')}",
            })

print("Schema errors:", len(schema_errors))

if schema_errors:
    display(pd.DataFrame(schema_errors).head(20))
else:
    print("OK: file messages JSONL hợp lệ để train dạng chat SFT.")

Schema errors: 0
OK: file messages JSONL hợp lệ để train dạng chat SFT.


In [4]:
rows = []

for r in records:
    messages = r.get("messages", [])

    user_msg = ""
    assistant_msg = ""

    for m in messages:
        if m.get("role") == "user":
            user_msg = m.get("content", "")
        elif m.get("role") == "assistant":
            assistant_msg = m.get("content", "")

    rows.append({
        "id": r.get("id", ""),
        "type": r.get("type", ""),
        "num_messages": len(messages),
        "user_preview": user_msg[:700],
        "assistant_preview": assistant_msg[:700],
    })

df_msg = pd.DataFrame(rows)

print("Shape:", df_msg.shape)
display(df_msg)

Shape: (20, 5)


,id,type,num_messages,user_preview,assistant_preview
0,sample_0001,noisy_context,2,Câu hỏi:\nChiến thắng Bạch Đằng năm 938 gắn vớ...,Nguồn được dùng: [hf_wikipedia_ngô_quyền_0000_...
1,sample_0002,noisy_context,2,Câu hỏi:\nBạch Đằng năm 938 đã đánh bại quân x...,Nguồn được dùng: [hf_wikipedia_ngô_quyền_0000_...
2,sample_0003,noisy_context,2,Câu hỏi:\nÝ nghĩa lịch sử của chiến thắng Bạch...,Nguồn được dùng: [hf_wikipedia_ngô_quyền_0000_...
3,sample_0004,noisy_context,2,"Câu hỏi:\nSau chiến thắng Bạch Đằng, Ngô Quyền...",Nguồn được dùng: [hf_wikipedia_ngô_quyền_0000_...
4,sample_0005,noisy_context,2,Câu hỏi:\nKiều Công Tiễn có vai trò gì trong b...,Nguồn được dùng: [hf_wikipedia_ngô_quyền_0001_...
5,sample_0006,noisy_context,2,Câu hỏi:\nNgô Quyền đã dùng kế gì để đánh quân...,Nguồn được dùng: [hf_wikipedia_ngô_quyền_0002_...
6,sample_0007,noisy_context,2,Câu hỏi:\nLưu Hoằng Tháo trong trận Bạch Đằng ...,Nguồn được dùng: [hf_wikipedia_ngô_quyền_0002_...
7,sample_0008,noisy_context,2,Câu hỏi:\nNgô Quyền xưng vương và đóng đô ở đâ...,Nguồn được dùng: [hf_wikipedia_ngô_quyền_0001_...
8,sample_0009,noisy_context,2,Câu hỏi:\nVì sao Ngô Quyền không chọn Đại La l...,Nguồn được dùng: [hf_wikipedia_ngô_quyền_0003_...
9,sample_0010,noisy_context,2,Câu hỏi:\nTrận Bạch Đằng năm 1288 do ai chỉ hu...,Nguồn được dùng: [hf_wikipedia_trận_bạch_đằng_...


In [5]:
idx = 0  # đổi số này để xem sample khác

sample = records[idx]

print("=" * 120)
print("ID:", sample.get("id"))
print("TYPE:", sample.get("type"))
print("=" * 120)

for m in sample["messages"]:
    print("\n" + "-" * 120)
    print("ROLE:", m["role"])
    print("-" * 120)
    print(m["content"])

ID: sample_0001
TYPE: noisy_context

------------------------------------------------------------------------------------------------------------------------
ROLE: user
------------------------------------------------------------------------------------------------------------------------
Câu hỏi:
Chiến thắng Bạch Đằng năm 938 gắn với nhân vật nào?

Tài liệu tham khảo:
[hf_wikipedia_ngô_quyền_0000_af223d790816] Ngô Quyền
Ngô Quyền (; 17 tháng 4 năm 898 – 14 tháng 2 năm 944), còn được biết đến với tên gọi Tiền Ngô Vương () là vị vua đầu tiên của nhà Ngô trong lịch sử Việt Nam. Năm 938, ông là người lãnh đạo nhân dân đánh bại quân Nam Hán trong trận Bạch Đằng, chính thức kết thúc gần một ngàn năm Bắc thuộc, mở ra một thời kì độc lập lâu dài của Việt Nam. Sau chiến thắng này, ông lên ngôi vua, lập ra nhà Ngô, trị vì từ năm 939 đến năm 944. Ngô Quyền nằm trong danh sách mười bốn anh hùng dân tộc Việt Nam. Phan Bội Châu xem ông là vị Tổ Trung hưng của Việt Nam. Thân thế Ngô Quyền sinh ngày 

In [7]:
df_msg["type"].value_counts()
type_counts = df_msg["type"].value_counts()
type_percent = (type_counts / len(df_msg) * 100).round(2)

summary_df = pd.DataFrame({
    "count": type_counts,
    "percent": type_percent,
})

display(summary_df)

,count,percent
type,,
noisy_context,13,65.0
grounded_qa,4,20.0
insufficient_context,2,10.0
false_premise,1,5.0


In [9]:
def simple_format_chat(messages):
    parts = []

    for m in messages:
        role = m["role"]
        content = m["content"]

        if role == "system":
            parts.append(f"<system>\n{content}\n</system>")
        elif role == "user":
            parts.append(f"<user>\n{content}\n</user>")
        elif role == "assistant":
            parts.append(f"<assistant>\n{content}\n</assistant>")

    return "\n\n".join(parts)

idx = 0

formatted = simple_format_chat(records[idx]["messages"])

print(formatted[:4000])

<user>
Câu hỏi:
Chiến thắng Bạch Đằng năm 938 gắn với nhân vật nào?

Tài liệu tham khảo:
[hf_wikipedia_ngô_quyền_0000_af223d790816] Ngô Quyền
Ngô Quyền (; 17 tháng 4 năm 898 – 14 tháng 2 năm 944), còn được biết đến với tên gọi Tiền Ngô Vương () là vị vua đầu tiên của nhà Ngô trong lịch sử Việt Nam. Năm 938, ông là người lãnh đạo nhân dân đánh bại quân Nam Hán trong trận Bạch Đằng, chính thức kết thúc gần một ngàn năm Bắc thuộc, mở ra một thời kì độc lập lâu dài của Việt Nam. Sau chiến thắng này, ông lên ngôi vua, lập ra nhà Ngô, trị vì từ năm 939 đến năm 944. Ngô Quyền nằm trong danh sách mười bốn anh hùng dân tộc Việt Nam. Phan Bội Châu xem ông là vị Tổ Trung hưng của Việt Nam. Thân thế Ngô Quyền sinh ngày 12 tháng 3 năm Đinh Tỵ (17 tháng 4 năm 898) trong một dòng họ hào trưởng có thế lực. Cha là Ngô Mân làm chức châu mục Đường Lâm. Ngô Quyền được sử sách mô tả là bậc anh hùng tuấn kiệt, "có trí dũng". Theo Đại Việt Sử ký Toàn thư: Sự nghiệp Bối cảnh Thời bấy giờ nhà Đường ở Trung Quố

In [10]:
length_rows = []

for r in records:
    user_text = ""
    assistant_text = ""

    for m in r["messages"]:
        if m["role"] == "user":
            user_text += m["content"]
        elif m["role"] == "assistant":
            assistant_text += m["content"]

    length_rows.append({
        "id": r.get("id", ""),
        "type": r.get("type", ""),
        "user_chars": len(user_text),
        "assistant_chars": len(assistant_text),
        "user_words": len(user_text.split()),
        "assistant_words": len(assistant_text.split()),
    })

df_len = pd.DataFrame(length_rows)

display(df_len)
display(df_len.describe())

,id,type,user_chars,assistant_chars,user_words,assistant_words
0,sample_0001,noisy_context,6940,211,1464,36
1,sample_0002,noisy_context,6914,195,1467,24
2,sample_0003,noisy_context,6922,197,1489,35
3,sample_0004,noisy_context,6928,169,1468,29
4,sample_0005,noisy_context,6957,240,1503,45
5,sample_0006,noisy_context,6931,274,1461,51
6,sample_0007,noisy_context,6928,211,1460,36
7,sample_0008,noisy_context,6906,196,1510,26
8,sample_0009,noisy_context,6909,352,1520,73
9,sample_0010,noisy_context,6911,265,1465,44


,user_chars,assistant_chars,user_words,assistant_words
count,20.000000,20.000000,20.000000,20.000000
mean,5786.200000,233.650000,1238.500000,39.950000
std,1895.007002,65.115505,398.165993,13.307991
min,2345.000000,143.000000,508.000000,23.000000
25%,4636.750000,195.750000,1014.250000,28.250000
50%,6918.000000,213.500000,1466.000000,37.000000
75%,6932.750000,269.500000,1473.250000,49.250000
max,6986.000000,363.000000,1520.000000,73.000000
